In [ ]:
import numpy as np
import pandas as pd
import re

ING_META_PATH = "ingredient_meta.csv"          
ING_EMB_PATH  = "ingredient_embeddings.npy"    

ingredient_meta = pd.read_csv(ING_META_PATH)
ingredient_embeddings = np.load(ING_EMB_PATH)

assert "ingredient_name" in ingredient_meta.columns
assert len(ingredient_meta) == ingredient_embeddings.shape[0]

EMBEDDING_DIM = ingredient_embeddings.shape[1]

# 정규화(가벼운 규칙) : 공백/특수문자/대괄호/괄호/중복공백 제거
def norm_ing(s: str) -> str:
    s = str(s)
    s = s.strip()
    s = s.lower()
    s = re.sub(r"[\[\]\(\)\{\}]", " ", s)
    s = re.sub(r"[^0-9a-z가-힣\s\-]", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s

# ingredient_name 정규화 컬럼
ingredient_meta["ingredient_name_norm"] = ingredient_meta["ingredient_name"].map(norm_ing)

# norm -> index 매핑 (동일 norm이 여러개면 첫번째 사용)
norm_to_idx = {}
for i, x in enumerate(ingredient_meta["ingredient_name_norm"].tolist()):
    if x and x not in norm_to_idx:
        norm_to_idx[x] = i

# (디버그용) 원문 -> idx (정규화 기준)
def find_exact_idx(ing_name: str):
    key = norm_ing(ing_name)
    return norm_to_idx.get(key, None)

성분 임베딩 로드 & 페르소나 성분 벡터

In [7]:

try:
    ingredient_meta = pd.read_csv("ingredient_meta.csv")
except FileNotFoundError:
    ingredient_meta = pd.DataFrame(columns=["ingredient_name"])

try:
    ingredient_embeddings = np.load("ingredient_embeddings.npy")
except FileNotFoundError:
    ingredient_embeddings = np.zeros((0, 8), dtype=np.float32)

if "ingredient_name" not in ingredient_meta.columns:
    ingredient_meta["ingredient_name"] = []

if ingredient_embeddings.shape[0] > 0:
    EMBEDDING_DIM = ingredient_embeddings.shape[1]
else:
    EMBEDDING_DIM = 8  # fallback

ingredient_embedding_dict = {
    str(row["ingredient_name"]).strip(): ingredient_embeddings[idx]
    for idx, row in ingredient_meta.iterrows()
    if idx < ingredient_embeddings.shape[0]
}

persona_ingredient_preference = {
    "persona_1": ["히알루론산", "시카"],
    "persona_2": ["나이아신아마이드"],
    "persona_3": ["인삼", "병풀"]
}

def mean_embedding(ingredients, emb_dict, dim):
    vecs = [emb_dict[i] for i in ingredients if i in emb_dict]
    return np.mean(vecs, axis=0) if len(vecs) > 0 else np.zeros(dim)

persona_ingredient_vector = {
    pid: mean_embedding(ings, ingredient_embedding_dict, EMBEDDING_DIM)
    for pid, ings in persona_ingredient_preference.items()
}

리스크 / 가격 수치 벡터

In [8]:
# [민감도, 트러블, 향 민감도, 가격 민감도]
persona_risk_price_vector = {
    "persona_1": [1.0, 0.9, 0.8, 0.3],
    "persona_2": [0.3, 0.2, 0.2, 0.9],
    "persona_3": [0.4, 0.3, 0.1, 0.2],
}

In [10]:

persona_ids = sorted(
    set(persona_tone.keys())
    & set(persona_ingredient_vector.keys())
    & set(persona_risk_price_vector.keys())
)


persona_final_vector = {}

for pid in persona_ids:
    tone_vec = np.array(persona_tone[pid], dtype=np.float32)
    ingredient_vec = np.array(persona_ingredient_vector[pid], dtype=np.float32)
    risk_price_vec = np.array(persona_risk_price_vector[pid], dtype=np.float32)

    final_vec = np.concatenate([tone_vec, ingredient_vec, risk_price_vec], axis=0)
    persona_final_vector[pid] = final_vec

In [11]:
rows = []
for pid, vec in persona_final_vector.items():
    rows.append({
        "persona_id": pid,
        "tone_vector": persona_tone[pid],
        "ingredient_vector": persona_ingredient_vector[pid].tolist(),
        "risk_price_vector": persona_risk_price_vector[pid],
        "final_vector": vec.tolist()
    })

persona_df = pd.DataFrame(rows)
persona_df.to_csv("persona_vectors.csv", index=False)

In [12]:
dims = [len(v) for v in persona_final_vector.values()]
print("vector dims:", dims)

from sklearn.metrics.pairwise import cosine_similarity

vecs = np.stack(list(persona_final_vector.values()))
cosine_similarity(vecs)

vector dims: [16, 16, 16]


array([[1.       , 0.5006559, 0.6998017],
       [0.5006559, 0.9999999, 0.5242197],
       [0.6998017, 0.5242197, 1.       ]], dtype=float32)